# CheckMaize - All-in-One Notebook

One notebook, one session, everything: download the data, train the five
models, pick the winner, and export the app-ready model.

**How to use:**

1. Upload this notebook to a **T4 GPU** runtime (colab.research.google.com ->
   Runtime -> Change runtime type -> T4 GPU).
2. Run the cells **in order, one at a time, top to bottom**.
3. Keep this tab open while cells run. Each cell prints what to expect.

**If the session drops:** reconnect, re-run Cells 1-5 (data + Drive restore),
then continue where you stopped. Finished models are restored from Google
Drive (browser Colab) and skipped, so you lose almost nothing.

In [ ]:
import os, importlib

REPO_URL = 'https://github.com/litcorp0/checkmaize.git'  # change if you fork the project

def repo_root():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            # The split CSVs are generated locally and committed to GitHub;
            # clean the untracked copies so the pull can update them cleanly.
            os.system(f'cd {repo} && git clean -fdq data/manifests && git checkout -q -- data/manifests')
            os.system(f'cd {repo} && git pull')
            return repo
        print('Repo not on this runtime yet. Cloning from GitHub...')
        result = os.system(f'git clone {REPO_URL} /content/checkmaize')
        if result != 0 or not os.path.exists(repo):
            print('Automatic clone failed. Likely causes:')
            print('  - the GitHub repo is private (make it public first), or')
            print('  - no internet on this runtime.')
            print('Manual fix - run this in a NEW cell, then re-run this cell:')
            print(f'  !git clone {REPO_URL} /content/checkmaize')
            print('Or drag the checkmaize folder into the Colab file explorer (into /content).')
            raise SystemExit
        return repo
    return os.path.abspath('..')

REPO = repo_root()
os.chdir(REPO)
print('Working in:', os.getcwd())

missing = []
for mod in ['numpy', 'PIL', 'pandas', 'yaml', 'sklearn', 'matplotlib', 'huggingface_hub', 'onnx', 'onnxruntime', 'onnxscript', 'pytest']:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print('installing missing packages:', missing)
    !pip install -q -r requirements.txt
    print('dependencies installed')
else:
    print('dependencies OK')

try:
    import torch
    print('torch:', torch.__version__, '| GPU available:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        print('WARNING: no GPU on this runtime. In browser Colab: Runtime ->')
        print('Change runtime type -> T4 GPU, then re-run this cell. Training')
        print('on CPU takes many hours.')
except ImportError:
    print('torch is not installed in this environment. Install it (GPU build) before training.')


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())

import shutil, zipfile
from huggingface_hub import hf_hub_download

REPO_ID = 'mohanty/PlantVillage'

dst = 'data/raw/plantvillage'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)

CLASS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_': 'Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'healthy',
}

print('downloading data.zip (~2 GB, 10-30 minutes)...')
data_zip = hf_hub_download(REPO_ID, 'data.zip', repo_type='dataset')
print('downloaded:', data_zip)

train_txt = hf_hub_download(REPO_ID, 'splits/color_train.txt', repo_type='dataset')
test_txt = hf_hub_download(REPO_ID, 'splits/color_test.txt', repo_type='dataset')

extract_dir = '/content/pv_extract'
shutil.rmtree(extract_dir, ignore_errors=True)
with zipfile.ZipFile(data_zip) as z:
    z.extractall(extract_dir)

raw_root = None
for root, dirs, files in os.walk(extract_dir):
    if os.path.basename(root) == 'raw' and 'color' in dirs:
        raw_root = root
        break
if raw_root is None:
    raise SystemExit('could not find raw/color inside data.zip - unexpected layout')

def leaf_id_for(file_name):
    ident = file_name.split('___')[-1]
    ident = ident.split('copy')[0]
    for ext in ('.jpg', '.JPG', '.jpeg', '.png', '.PNG'):
        ident = ident.replace(ext, '')
    return ident.strip()

copied = 0
missing = 0
for txt in [train_txt, test_txt]:
    with open(txt) as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            parts = rel.split('/')
            if len(parts) < 4 or parts[0] != 'raw' or parts[1] != 'color':
                continue
            class_dir = parts[2]
            if class_dir not in CLASS_MAP:
                continue
            file_name = parts[3]
            src = os.path.join(raw_root, *parts[1:])
            if not os.path.exists(src):
                missing += 1
                continue
            leaf_id = leaf_id_for(file_name)
            out_dir = os.path.join(dst, CLASS_MAP[class_dir])
            os.makedirs(out_dir, exist_ok=True)
            shutil.copyfile(src, os.path.join(out_dir, f'{leaf_id}__{file_name}'))
            copied += 1

print(f'plantvillage extraction done: {copied} maize images copied ({missing} missing)')


In [ ]:
import os
os.chdir('/content')

# (If you are in BROWSER Colab and already uploaded a dataset zip to /content
#  yourself, this cell will find and use it automatically - nothing to do here.)

# ---- Download the Ghana dataset directly from Kaggle ----
# NOTE: your credentials are filled in below. DO NOT commit/push this notebook
# while they are here - tell your assistant when done so they can scrub them.
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'
os.environ['KAGGLE_API_TOKEN'] = 'YOUR_KAGGLE_KEY'
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download('nirmalsankalana/crop-pest-and-disease-detection')
print('downloaded and extracted to:', path)

import os, zipfile, shutil, glob
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())

NEEDED = {'Leaf blight': 'Leaf blight', 'Leaf spot': 'Leaf spot', 'Healthy': 'Healthy'}
NEEDED_LOWER = {k.lower(): k for k in NEEDED}

candidates = []
for name in ['archive.zip', 'Raw Data.zip', 'crop-pest-and-disease-detection.zip']:
    p = f'/content/{name}'
    if os.path.exists(p):
        candidates.append(p)
if not candidates:
    candidates = sorted(z for z in glob.glob('/content/*.zip') if 'checkmaize' not in z)

kaggle_dirs = sorted(glob.glob('/root/.cache/kagglehub/datasets/nirmalsankalana/crop-pest-and-disease-detection/versions/*'))
# Newer kagglehub versions use a Colab cache (e.g. /kaggle/input/...); the
# `path` returned by dataset_download is the real location - check it first.
kaggle_locs = [path] + [d for d in kaggle_dirs if d != path]

extract_dir = '/content/ccmt_extract'
shutil.rmtree(extract_dir, ignore_errors=True)

if candidates:
    zip_path = candidates[0]
    print('Using:', os.path.basename(zip_path))
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
elif kaggle_locs:
    loc = kaggle_locs[0]
    if os.path.isfile(loc) and loc.lower().endswith('.zip'):
        print('Using kagglehub zip:', loc)
        with zipfile.ZipFile(loc) as z:
            z.extractall(extract_dir)
    else:
        extract_dir = loc
        print('Using kagglehub download:', extract_dir)
else:
    print('No Ghana dataset found on the cloud machine yet.')
    print('The Kaggle download reported a location but no files were found.')
    print('Re-run this cell; if it keeps failing, tell your assistant.')
    raise SystemExit

dst = 'data/raw/ccmt_ghana'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
copied = {}

def copy_dir(src, key):
    target = os.path.join(dst, NEEDED[key])
    shutil.copytree(src, target)
    copied[key] = len(os.listdir(target))

nested_maize = None
for root, dirs, files in os.walk(extract_dir):
    if 'Maize' in dirs:
        nested_maize = os.path.join(root, 'Maize')
        break

if nested_maize and any(k.lower() in NEEDED_LOWER for k in os.listdir(nested_maize)):
    for sub in sorted(os.listdir(nested_maize)):
        if sub.lower() in NEEDED_LOWER:
            copy_dir(os.path.join(nested_maize, sub), NEEDED_LOWER[sub.lower()])
            print(sub, copied[NEEDED_LOWER[sub.lower()]])
else:
    for folder in sorted(os.listdir(extract_dir)):
        full = os.path.join(extract_dir, folder)
        if folder.startswith('Maize ') and os.path.isdir(full):
            key = folder[len('Maize '):].lower()
            if key in NEEDED_LOWER:
                copy_dir(full, NEEDED_LOWER[key])
                print(key, copied[NEEDED_LOWER[key]])

missing = [k for k in NEEDED if k not in copied]
assert not missing, f'Could not find classes {missing} in the dataset'
print('ccmt_ghana ready:', copied)


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
!python -m data.make_manifest
!python -m data.make_splits
!python -m pytest data/tests -v


shutil.make_archive('/content/splits', 'zip', 'data/manifests')
try:
    from google.colab import files
    files.download('/content/splits.zip')
    print('download started (browser Colab)')
except Exception:
    print('VS Code mode: the file is at /content/splits.zip')
    print('Drag it from the file explorer onto your computer, unzip it, and put the')
    print('CSV files into: checkmaize/data/manifests/')


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
!python -m pytest training/tests benchmarks/tests -v


In [ ]:
import os, shutil, glob

# OPTIONAL but recommended (browser Colab): back up every finished model to your
# Google Drive, so a dropped session never costs you training progress.
# In VS Code or without Drive, this cell skips itself harmlessly.

if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Drive mounted')
    except Exception:
        print('Drive not available - continuing without backup (training still works).')

bak = '/content/drive/MyDrive/checkmaize_backup/runs'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(bak, exist_ok=True)
    restored = []
    for mdir in sorted(glob.glob(bak + '/*')):
        model = os.path.basename(mdir)
        dst = '/content/checkmaize/artifacts/runs/' + model
        os.makedirs(dst, exist_ok=True)
        for name in ('best.pt', 'metrics.json', 'model.onnx'):
            s = os.path.join(mdir, name)
            if os.path.exists(s):
                shutil.copy(s, os.path.join(dst, name))
        restored.append(model)
    if restored:
        print('Restored training progress from Drive for:', ', '.join(restored))
    else:
        print('No previous progress found in Drive.')


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
print('Training all five models. This takes 2-3 hours - keep this tab open.')
print('Each finished model is backed up automatically (Drive if mounted).')
print('If the session drops: reconnect, re-run Cells 1-5, then this cell -')
print('finished models are skipped, so you lose almost nothing.')
!python -m benchmarks.compare
print()
print('Scoreboard above. The `accuracy` column is the score on the Ghana-only')
print('test set. Pick the winner: highest accuracy with onnx_bytes under')
print('~20 MB (usually efficientnet_b0). Remember its name for Cell 10.')


In [ ]:
import os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
shutil.make_archive('/content/report', 'zip', 'benchmarks/report')
shutil.make_archive('/content/runs', 'zip', 'artifacts/runs')
try:
    from google.colab import files
    files.download('/content/report.zip')
    files.download('/content/runs.zip')
    print('downloads started (browser Colab)')
except Exception:
    print('VS Code mode: the files are at /content/report.zip and /content/runs.zip')
    print('Drag them from the file explorer onto your computer.')
    print('Unzip report.zip into checkmaize/benchmarks/report/')
    print('Unzip runs.zip into checkmaize/artifacts/runs/')


## Pick the winner

Look at the scoreboard printed by the previous cell. Choose the model with
the highest `accuracy` whose `onnx_bytes` is reasonable for a phone (under
~20 MB). Usually this is `efficientnet_b0`. You will type its name in the
next cell.

In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
winner = 'efficientnet_b0'
assert os.path.exists(f'artifacts/runs/{winner}/best.pt'), f'no checkpoint for {winner}'
print('winner:', winner)


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
try:
    winner
except NameError:
    winner = 'efficientnet_b0'  # SET THIS to the model chosen from Notebook 02's scoreboard

!python -m inference.export --checkpoint artifacts/runs/{winner}/best.pt --out artifacts/runs/{winner}/model.onnx
!python -m inference.quantize --fp32 artifacts/runs/{winner}/model.onnx --out artifacts/model_int8.onnx
!python -m inference.verify --checkpoint artifacts/runs/{winner}/best.pt --fp32 artifacts/runs/{winner}/model.onnx --int8 artifacts/model_int8.onnx


In [ ]:
import json, os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
try:
    winner
except NameError:
    winner = 'efficientnet_b0'  # SET THIS to the model chosen from Notebook 02's scoreboard

labels = ['common_rust', 'gray_leaf_spot', 'northern_leaf_blight', 'healthy']
with open('artifacts/labels.json', 'w') as f:
    json.dump(labels, f)
with open('artifacts/runs/' + winner + '/metrics.json') as f:
    m = json.load(f)
with open('inference/verify_report.json') as f:
    v = json.load(f)
if v['ship_int8']:
    shipped_file = 'artifacts/model_int8.onnx'
else:
    shipped_file = 'artifacts/model.onnx'
    shutil.copy('artifacts/runs/' + winner + '/model.onnx', shipped_file)
metrics = {
    'model': winner,
    'test_accuracy': m['accuracy'],
    'macro_f1': m['macro_f1'],
    'onnx_bytes': os.path.getsize(shipped_file),
    'int8_test_accuracy': v['int8_accuracy'],
    'shipped': 'int8' if v['ship_int8'] else 'fp32',
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


In [ ]:
import os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
shutil.make_archive('/content/artifacts', 'zip', 'artifacts')
shutil.make_archive('/content/fixtures', 'zip', 'app/src/ml/__tests__/fixtures')
try:
    from google.colab import files
    files.download('/content/artifacts.zip')
    files.download('/content/fixtures.zip')
    files.download('docs/onnx-contract.md')
    print('downloads started (browser Colab)')
except Exception:
    print('VS Code mode: files are at:')
    print('  /content/artifacts.zip        -> unzip, copy the 3 files into checkmaize/app/assets/model/')
    print('  /content/fixtures.zip         -> unzip, copy the 2 files into checkmaize/app/src/ml/__tests__/fixtures/ (replace)')
    print('  docs/onnx-contract.md (in the repo) -> it is already in place')
    print('Drag the zips from the file explorer onto your computer.')
